# Stable Video Infinity (SVI) on Google Colab 🎬♾️

Runs the official **[vita-epfl/Stable-Video-Infinity](https://github.com/vita-epfl/Stable-Video-Infinity)** repo — infinite-length video generation built on **Wan 2.1 I2V 14B**.

## ⚠️ READ FIRST — Hardware requirements
| Runtime | Will it work? |
|---|---|
| Free Colab T4 (16 GB) | ❌ OOM at load (model is 14 B params) |
| Colab Pro L4 (22 GB) | ⚠️ Marginal with heavy offload |
| Colab Pro V100 (16 GB) | ❌ OOM |
| **Colab Pro+ A100 40 GB** | ✅ Recommended |

Disk: needs **~55 GB free** for weights. Use **Colab Pro+ → High-RAM A100**.

## You also need
1. A **Hugging Face account** with a **fine-grained access token** (Settings → Access Tokens).
2. Accept the license on the Wan model page: <https://huggingface.co/Wan-AI/Wan2.1-I2V-14B-480P>
3. (Optional) Accept on <https://huggingface.co/vita-video-gen/svi-model>

## Plan
1. Check GPU / disk
2. Clone repo & install deps
3. Login to HF, download Wan 2.1 I2V 14B + SVI LoRA (~50 GB)
4. Run `test_svi.py` on the provided demo image+prompt
5. Display the generated long video

## 0. Sanity checks

In [ ]:
!nvidia-smi || echo '❌ No GPU. Runtime → Change runtime type → GPU (A100 ideally)'
!df -h /content | tail -1
import torch
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'VRAM: {vram:.1f} GB')
    if vram < 22:
        print('⚠️  Less than 22 GB VRAM — this will very likely OOM.')

## 1. Clone repo & install dependencies

In [ ]:
%cd /content
![ -d Stable-Video-Infinity ] || git clone https://github.com/vita-epfl/Stable-Video-Infinity.git
%cd /content/Stable-Video-Infinity
!pip install -q -e .
# Repo requirements
!pip install -q -r requirements.txt
# flash-attn (this is the slow & finicky one)
!pip install -q flash_attn==2.8.0.post2 --no-build-isolation || echo '⚠️ flash-attn failed; trying prebuilt wheel'
!apt-get -q install -y ffmpeg > /dev/null
print('✅ install done')

## 2. Hugging Face login

Paste your HF token below. Get one at <https://huggingface.co/settings/tokens> (read access is enough).

In [ ]:
from huggingface_hub import login
import getpass
token = getpass.getpass('Paste your HF token: ')
login(token=token, add_to_git_credential=False)
print('✅ logged in')

## 3. Download model weights (~50 GB, can take 10–25 min)

Downloads:
- **Wan 2.1 I2V 14B 480P** (base) → `weights/Wan2.1-I2V-14B-480P/`
- **SVI-Shot LoRA** (the lightest SVI variant, recommended to start) → `weights/Stable-Video-Infinity/version-1.0/svi-shot.safetensors`

In [ ]:
%cd /content/Stable-Video-Infinity
!huggingface-cli download Wan-AI/Wan2.1-I2V-14B-480P \
    --local-dir ./weights/Wan2.1-I2V-14B-480P --max-workers 8
!huggingface-cli download vita-video-gen/svi-model \
    version-1.0/svi-shot.safetensors \
    --local-dir ./weights/Stable-Video-Infinity
!ls -lh weights/Wan2.1-I2V-14B-480P/ | head
!ls -lh weights/Stable-Video-Infinity/version-1.0/
!df -h /content | tail -1

## 4. (Optional) Use your own image + prompts

Skip this cell to use the demo image included in the repo (`data/toy_test/shot/frame.jpg`).

To use your own:
- Upload an image (the first frame) via the Colab file panel.
- Edit `MY_PROMPTS` below.

In [ ]:
import os, shutil
USE_CUSTOM = False  # set True to use your own image+prompts
MY_IMAGE   = '/content/my_first_frame.jpg'   # upload here
MY_PROMPTS = [
    'The subject walks slowly forward, cinematic camera follows.',
    'The subject continues walking, camera pans slightly.',
    'The subject stops and looks around.',
]

if USE_CUSTOM:
    assert os.path.exists(MY_IMAGE), f'put your image at {MY_IMAGE}'
    custom_dir = '/content/Stable-Video-Infinity/data/toy_test/my_shot'
    os.makedirs(custom_dir, exist_ok=True)
    shutil.copy(MY_IMAGE, f'{custom_dir}/frame.jpg')
    with open(f'{custom_dir}/prompt.txt', 'w') as f:
        f.write('prompts = ' + repr(MY_PROMPTS))
    REF_IMG  = f'{custom_dir}/frame.jpg'
    PROMPT_F = f'{custom_dir}/prompt.txt'
else:
    REF_IMG  = 'data/toy_test/shot/frame.jpg'
    PROMPT_F = 'data/toy_test/shot/prompt.txt'
print('reference image:', REF_IMG)
print('prompt file    :', PROMPT_F)

## 5. Run SVI-Shot inference

- `--num_clips 4`  → output ≈ 4 × 5 s = **20 s** (raise it for longer videos; SVI handles infinite length but each clip takes time)
- `--num_steps 50` → diffusion steps (lower = faster, worse quality)
- `--num_persistent_param_in_dit 3000000000` → keep only 3 B params on GPU at once → enables A100 40 GB. Lower it more (e.g. `1500000000`) for L4 22 GB; raise to `6000000000` on A100 80 GB.

Expect roughly **3–8 min per 5 s clip** on A100 40 GB.

In [ ]:
%cd /content/Stable-Video-Infinity
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

!python test_svi.py \
    --output videos/svi_shot/ \
    --dit_root ./weights/Wan2.1-I2V-14B-480P/ \
    --extra_module_root ./weights/Stable-Video-Infinity/version-1.0/svi-shot.safetensors \
    --ref_image_path {REF_IMG} \
    --prompt_path   {PROMPT_F} \
    --ref_pad_num -1 \
    --cfg_scale_text 5.0 \
    --num_motion_frames 1 \
    --num_clips 4 \
    --num_steps 50 \
    --use_first_prompt_only \
    --num_persistent_param_in_dit 3000000000

## 6. Show the generated video

In [ ]:
import glob, os
from IPython.display import Video, display
vids = sorted(glob.glob('/content/Stable-Video-Infinity/videos/svi_shot/**/*.mp4', recursive=True))
print(f'found {len(vids)} video(s):')
for v in vids: print(' ', v, f'({os.path.getsize(v)/1024/1024:.1f} MB)')
if vids:
    display(Video(vids[-1], embed=True, width=640))
else:
    print('⚠️ no video produced — check the inference log above for errors')

## 7. (Optional) Try other SVI variants

Once SVI-Shot works, download more LoRAs and swap `--extra_module_root`:

```bash
# multi-scene film (uses 5 motion frames)
!huggingface-cli download vita-video-gen/svi-model version-1.0/svi-film-opt-10212025.safetensors \
    --local-dir ./weights/Stable-Video-Infinity
# cartoon (Tom & Jerry style)
!huggingface-cli download vita-video-gen/svi-model version-1.0/svi-tom.safetensors \
    --local-dir ./weights/Stable-Video-Infinity
# latest improved (SVI 2.0)
!huggingface-cli download vita-video-gen/svi-model version-2.0/SVI_Wan2.1-I2V-14B_lora_v2.0.safetensors \
    --local-dir ./weights/Stable-Video-Infinity
```

Then re-run cell 5 with the new path, and for `svi-film*` use `--num_motion_frames 5` (instead of `1`).

## 8. Common errors
| Error | Fix |
|---|---|
| `CUDA out of memory` | Lower `--num_persistent_param_in_dit` (try 1500000000), or `--num_steps 30`, or smaller resolution in `test_svi.py` (`height=320, width=576`). If still OOM → you need A100 80 GB. |
| `flash_attn` install fails | Comment out the flash-attn line; SVI also runs (slower) without it after small edits — see repo issue #3. |
| `403 / gated repo` | Accept the Wan 2.1 license on its HF page, then re-login. |
| Out of disk | Use Colab Pro+ (200 GB) or delete the dataset cache. |
| `KeyError 'prompts'` | Your `prompt.txt` must define `prompts = ["...", "..."]` as Python list. |